# 재무비율 기반 분식회계 확률 예측 모델
감사대상회사의 재무비율을 입력하면, 딥러닝 모델이 분식회계 확률(%)을 출력합니다.

## 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(1)

## 2. 데이터 불러오기
고정된 학습용 데이터(`training_data_final_v2.csv`)를 읽어옵니다.

In [ ]:
df = pd.read_csv('./training_data_final_v2.csv')
df.head()

In [ ]:
df.info()

## 3. 데이터 분석 (라벨 분포 확인)

In [ ]:
import seaborn as sns
sns.countplot(data=df, x='label')
plt.title('label distribution (0=정상, 1=분식)')
plt.show()

## 4. 이상치 처리
재무비율 지표들은 극단값이 있을 수 있으므로, 상하위 1% 값을 경계값으로 조정합니다 (클리핑).

In [ ]:
FEATURES = ['DSRI', 'GMI', 'AQI', 'SGI', 'DEPI', 'SGAI', 'LVGI', 'TATA']

for col in FEATURES:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=lower, upper=upper)

df[FEATURES].describe()

## 5. 결측치 확인

In [ ]:
print('결측치 개수\n', df[FEATURES].isnull().sum())

## 6. 훈련/검증 데이터셋 분리
- 피처(X): 재무비율 8개
- 레이블(y): label (0=정상, 1=분식)
- 비율: 80:20, 난수시드 42

In [ ]:
X = df[FEATURES]
y = df['label']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_valid.shape)

## 7. 스케일링 (RobustScaler)
이상치 영향을 줄이기 위해 RobustScaler를 사용합니다.

In [ ]:
rs = RobustScaler()
X_train_scaled = rs.fit_transform(X_train)
X_valid_scaled = rs.transform(X_valid)

round(np.max(X_valid_scaled))

## 8. 딥러닝 모델 구성
- 히든레이어 activation: relu
- 출력레이어 activation: sigmoid (확률 출력, 0~1 사이 값)
- EarlyStopping: val_loss 기준 9 epoch 개선 없으면 중단
- optimizer: adam, loss: binary_crossentropy, metrics: accuracy

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

estop = EarlyStopping(monitor='val_loss', patience=9)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 분식(1)기업 수가 정상(0)기업보다 훨씬 적으므로, 분식 클래스에 가중치를 부여
# WEIGHT_STRENGTH: 1.0=완전 균형(원래 비율 그대로), 0.5=절반 완화, 0=가중치 없음(원래 비율 무시)
WEIGHT_STRENGTH = 0.5

n_neg = sum(y_train == 0)
n_pos = sum(y_train == 1)
raw_ratio = n_neg / n_pos
adjusted_ratio = 1 + (raw_ratio - 1) * WEIGHT_STRENGTH
class_weight = {0: 1.0, 1: adjusted_ratio}
print(f'원래 비율: 1:{raw_ratio:.2f} -> 완화된 가중치: {class_weight}')

history = model.fit(
    X_train_scaled, y_train,
    batch_size=4, epochs=100,
    validation_data=(X_valid_scaled, y_valid),
    callbacks=[estop],
    class_weight=class_weight,
    verbose=1
)

## 9. 학습 곡선 시각화

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['loss', 'val_loss'])
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label='acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 10. 모델 성능 평가

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_pred_proba = model.predict(X_valid_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

acc = accuracy_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred)

print('정확도(Accuracy):', acc)
print('F1-Score:', f1)

## 11. 새 감사대상회사 재무제표 자동 조회 -> 비율 계산 -> 분식 확률 예측
8개 지표를 직접 입력하는 대신, **회사의 corp_code와 회계연도만 입력**하면
DART API로 재무제표를 자동으로 가져와서 비율을 계산하고 확률까지 출력합니다.

In [ ]:
import requests

API_KEY = '여기에_본인_DART_API_키를_입력하세요'  # 본인 DART API 키
BASE_URL = 'https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json'

ACCOUNT_KEYWORDS = {
    'sales': ['매출액', '수익(매출액)', '영업수익'],
    'cogs': ['매출원가'],
    'receivables': ['매출채권', '외상매출금'],
    'current_assets': ['유동자산'],
    'ppe': ['유형자산'],
    'total_assets': ['자산총계'],
    'sga': ['판매비와관리비', '판매비와 관리비'],
    'total_liabilities': ['부채총계'],
    'current_liabilities': ['유동부채'],
    'cash': ['현금및현금성자산'],
    'depreciation': ['감가상각비'],
}


def find_account_value(items, keywords, amount_key):
    for kw in keywords:
        for it in items:
            if kw in it.get('account_nm', ''):
                val = it.get(amount_key, '')
                val = val.replace(',', '') if val else ''
                if val not in ('', None):
                    try:
                        return float(val)
                    except ValueError:
                        continue
    return None


def fetch_financial_year(corp_code, year, fs_div='CFS'):
    params = {
        'crtfc_key': API_KEY,
        'corp_code': corp_code,
        'bsns_year': str(year),
        'reprt_code': '11011',
        'fs_div': fs_div,
    }
    resp = requests.get(BASE_URL, params=params, timeout=15)
    data = resp.json()
    if data.get('status') != '000':
        return None, None
    items = data.get('list', [])
    t_data, t1_data = {}, {}
    for key, kws in ACCOUNT_KEYWORDS.items():
        t_data[key] = find_account_value(items, kws, 'thstrm_amount')
        t1_data[key] = find_account_value(items, kws, 'frmtrm_amount')
    return t_data, t1_data


def compute_ratios(t, t1):
    # DSRI
    if t['receivables'] is not None and t1['receivables'] not in (None, 0) and t['sales'] and t1['sales']:
        dsri = (t['receivables'] / t['sales']) / (t1['receivables'] / t1['sales'])
    else:
        dsri = 1.0  # 매출채권 정보 없음 -> 중립값

    # GMI (매출원가가 없는 서비스/플랫폼 기업 대응)
    if t['cogs'] is not None and t1['cogs'] is not None:
        gm_t = (t['sales'] - t['cogs']) / t['sales']
        gm_t1 = (t1['sales'] - t1['cogs']) / t1['sales']
        gmi = gm_t1 / gm_t
    else:
        gmi = 1.0  # 매출원가 정보 없음 -> 중립값

    # AQI
    if None not in (t['current_assets'], t['ppe'], t['total_assets'],
                     t1['current_assets'], t1['ppe'], t1['total_assets']):
        aqi_t = 1 - (t['current_assets'] + t['ppe']) / t['total_assets']
        aqi_t1 = 1 - (t1['current_assets'] + t1['ppe']) / t1['total_assets']
        aqi = aqi_t / aqi_t1 if aqi_t1 != 0 else 1.0
    else:
        aqi = 1.0

    # SGI
    sgi = t['sales'] / t1['sales'] if t['sales'] and t1['sales'] else 1.0

    # DEPI (감가상각비는 원래도 결측이 잦은 항목)
    if t['depreciation'] and t1['depreciation'] and t['ppe'] and t1['ppe']:
        dep_rate_t = t['depreciation'] / (t['depreciation'] + t['ppe'])
        dep_rate_t1 = t1['depreciation'] / (t1['depreciation'] + t1['ppe'])
        depi = dep_rate_t1 / dep_rate_t
    else:
        depi = 1.0

    # SGAI
    if t['sga'] is not None and t1['sga'] is not None and t['sales'] and t1['sales']:
        sgai = (t['sga'] / t['sales']) / (t1['sga'] / t1['sales'])
    else:
        sgai = 1.0

    # LVGI
    if None not in (t['total_liabilities'], t['total_assets'], t1['total_liabilities'], t1['total_assets']):
        lvgi = (t['total_liabilities'] / t['total_assets']) / (t1['total_liabilities'] / t1['total_assets'])
    else:
        lvgi = 1.0

    # TATA
    if None not in (t['current_assets'], t['cash'], t['current_liabilities'], t['total_assets'],
                     t1['current_assets'], t1['cash'], t1['current_liabilities']):
        dep_for_tata = t['depreciation'] if t['depreciation'] else 0
        wc_t = (t['current_assets'] - t['cash']) - t['current_liabilities']
        wc_t1 = (t1['current_assets'] - t1['cash']) - t1['current_liabilities']
        tata = (wc_t - wc_t1 - dep_for_tata) / t['total_assets']
    else:
        tata = 0.0

    return {
        'DSRI': dsri, 'GMI': gmi, 'AQI': aqi, 'SGI': sgi,
        'DEPI': depi, 'SGAI': sgai, 'LVGI': lvgi, 'TATA': tata,
    }

### 회사 corp_code + 회계연도만 입력하세요
corp_code는 DART 홈페이지에서 회사명 검색 후 확인하거나, `dart_corp_code_all.csv`에서 찾을 수 있습니다.

In [ ]:
# 예시: 삼성전자 corp_code, 2023년 -> 실제 감사대상회사 정보로 교체해서 사용
target_corp_code = '00126380'
target_year = 2023

t, t1 = fetch_financial_year(target_corp_code, target_year, fs_div='CFS')
if t is None or all(v is None for v in t.values()):
    t, t1 = fetch_financial_year(target_corp_code, target_year, fs_div='OFS')

if t is None:
    print('데이터 조회 실패: corp_code 또는 연도를 확인하세요 (비상장/사업보고서 미제출 기업일 수 있음)')
else:
    ratios = compute_ratios(t, t1)
    print('계산된 재무비율:')
    for k, v in ratios.items():
        print(f'  {k}: {round(v, 4)}')

    new_company = pd.DataFrame([ratios])
    new_company_scaled = rs.transform(new_company)
    prob = model.predict(new_company_scaled)[0][0]
    print(f'\n분식회계 확률: {round(prob * 100, 2)}%')